# LangGraph Short-Term Memory with Azure Cosmos DB

This notebook demonstrates how to add **short-term memory** (conversation persistence) to a LangGraph agent using **Azure Cosmos DB** as the checkpointer backend.

## What is Short-Term Memory?

Short-term memory in LangGraph is implemented via **checkpointers**, which save a snapshot of the graph state at every step. When you assign a `thread_id` to a conversation, the checkpointer persists messages across invocations — enabling multi-turn conversations where the agent remembers what was said earlier.

## Why Azure Cosmos DB?

While `InMemorySaver` works for prototyping, it loses all state when the process restarts. For production workloads, you need a durable backend. Azure Cosmos DB provides:

- **Global distribution** — low-latency access from anywhere
- **Serverless pricing** — pay only for what you use
- **Automatic indexing** — no schema management needed
- **Microsoft Entra ID support** — secure, keyless authentication

## Architecture

```
User ──► LangGraph Agent ──► Self-hosted LLM (vLLM on ACA)
              │
              ▼
        CosmosDB Checkpointer
        (persists conversation state per thread)
```

## References

- [LangGraph Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [LangGraph Memory](https://docs.langchain.com/oss/python/langgraph/add-memory)
- [langchain-azure-cosmosdb](https://pypi.org/project/langchain-azure-cosmosdb/)

In [1]:
%pip install langchain langgraph langchain-openai langchain-azure-cosmosdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Get Endpoints

Retrieve the FQDN of the self-hosted LLM and the Cosmos DB connection details from Terraform outputs.

In [1]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

cosmosdb_endpoint = ! terraform output -raw cosmosdb_endpoint
cosmosdb_endpoint = cosmosdb_endpoint.n
print("CosmosDB Endpoint:", cosmosdb_endpoint)

cosmosdb_key = ! terraform output -raw cosmosdb_key
cosmosdb_key = cosmosdb_key.n
print("CosmosDB Key:", cosmosdb_key[:10] + "...")

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
CosmosDB Endpoint: https://cosmosdb-555.documents.azure.com:443/
CosmosDB Key: BG4iu5guTg...


## 3. Set Up the LLM Model

Create a `ChatOpenAI` model pointing at the vLLM-compatible endpoint running on Azure Container Apps.

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
    streaming=True,
    max_completion_tokens=512
)

## 4. Create the CosmosDB Checkpointer

The `CosmosDBSaverSync` checkpointer from `langchain-azure-cosmosdb` persists LangGraph state (messages, tool calls, etc.) into Azure Cosmos DB.

This is the key ingredient for **short-term memory**: every message in a conversation thread is saved to CosmosDB, so the agent can recall earlier messages in the same thread.

In [3]:
from langchain_azure_cosmosdb import CosmosDBSaverSync

checkpointer = CosmosDBSaverSync(
    database_name="langgraph-memory",
    container_name="checkpoints",
    endpoint=cosmosdb_endpoint,
    key=cosmosdb_key,
)

print("CosmosDB checkpointer ready")

CosmosDB checkpointer ready


## 5. Create an Agent with Memory

Compile a LangGraph agent with the CosmosDB checkpointer. Every invocation with the same `thread_id` will share conversation history — the agent "remembers" what was said before.

In [4]:
from langchain.agents import create_agent

agent = create_agent(model, checkpointer=checkpointer)
print("Agent with CosmosDB memory ready")

Agent with CosmosDB memory ready


## 6. Multi-Turn Conversation (Same Thread)

Now let's test short-term memory. We send two messages on the same `thread_id`:
1. **"Hi! I'm Alice."** — introduce ourselves
2. **"What's my name?"** — the agent should remember "Alice" from the previous message

All conversation state is persisted in CosmosDB between invocations.

In [6]:
from langchain_core.messages import HumanMessage

# Use a unique thread_id — all messages with the same thread_id share memory
config = {"configurable": {"thread_id": "demo-thread-1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Hi! I'm Alice and I live in Amsterdam.")]},
    config=config,
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! I'm Alice and I live in Amsterdam.
================================== Ai Message ==================================

Hi Alice! It's nice to meet you. Amsterdam is such a beautiful city—I imagine you have some amazing canals and architecture right outside your door.

How is your day going so far?
================================ Human Message =================================

Hi! I'm Alice and I live in Amsterdam.
================================== Ai Message ==================================

Hello again, Alice! You mentioned that just a moment ago. It's nice to meet you! 

Since you're in Amsterdam, I'm curious—do you have a favorite spot in the city, or perhaps a favorite way to spend a weekend there?
================================ Human Message =================================

What's my name and where do I live?
================================== Ai Message ==================================


### Turn 2: Test Memory Recall

Ask the agent a question that requires recalling information from Turn 1. Because we use the **same `thread_id`**, the CosmosDB checkpointer loads the previous conversation history and the agent can answer correctly.

In [7]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's my name and where do I live?")]},
    config=config,  # Same thread_id → agent remembers Turn 1
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! I'm Alice and I live in Amsterdam.
================================== Ai Message ==================================

Hi Alice! It's nice to meet you. Amsterdam is such a beautiful city—I imagine you have some amazing canals and architecture right outside your door.

How is your day going so far?
================================ Human Message =================================

Hi! I'm Alice and I live in Amsterdam.
================================== Ai Message ==================================

Hello again, Alice! You mentioned that just a moment ago. It's nice to meet you! 

Since you're in Amsterdam, I'm curious—do you have a favorite spot in the city, or perhaps a favorite way to spend a weekend there?
================================ Human Message =================================

What's my name and where do I live?
================================== Ai Message ==================================


## 7. Thread Isolation

Start a **new thread** (`demo-thread-2`). The agent should **not** remember Alice's name because each thread has its own isolated conversation history in CosmosDB.

In [ ]:
config_new_thread = {"configurable": {"thread_id": "demo-thread-2"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Do you know my name?")]},
    config=config_new_thread,  # Different thread_id → fresh conversation
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Do you know my name?
================================== Ai Message ==================================

No, I do not know your name. As an AI, I don't have access to your personal identity or private information unless you choose to share it with me during our conversation.


## 8. Resume the Original Thread

Switch back to `demo-thread-1`. The agent should still remember Alice and Amsterdam because the full conversation history is persisted in CosmosDB.

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Can you remind me what we talked about earlier?")]},
    config=config,  # Back to demo-thread-1
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! I'm Alice and I live in Amsterdam.
================================== Ai Message ==================================

Hi Alice! It's nice to meet you. Amsterdam is such a beautiful city—I imagine you have some amazing canals and architecture right outside your door.

How is your day going so far?
================================ Human Message =================================

Hi! I'm Alice and I live in Amsterdam.
================================== Ai Message ==================================

Hello again, Alice! You mentioned that just a moment ago. It's nice to meet you! 

Since you're in Amsterdam, I'm curious—do you have a favorite spot in the city, or perhaps a favorite way to spend a weekend there?
================================ Human Message =================================

What's my name and where do I live?
================================== Ai Message ==================================


## 9. Check the data saved in CosmosDB

Finally, we can inspect the CosmosDB container to see the saved conversation state. Each message and tool call is stored as a separate document, all linked by the `thread_id`.

In [ ]:
# list all documents in the CosmosDB container to verify that conversation state is being saved
from azure.cosmos import CosmosClient

client = CosmosClient(url=cosmosdb_endpoint, credential=cosmosdb_key)
database = client.get_database_client("langgraph-memory")
container = database.get_container_client("checkpoints")

for item in container.read_all_items():
    print(item)

# show the messages unencrypted in CosmosDB


{'partition_key': 'checkpoint$demo-thread-1$$', 'id': 'checkpoint$demo-thread-1$$1f1483ca-6312-63a0-bfff-4456b3ba9de3', 'thread_id': 'demo-thread-1', 'checkpoint': 'h6F2BKJ0c9kgMjAyNi0wNS0wNVQwNDo0MToxNC44NjcxODMrMDA6MDCiaWTZJDFmMTQ4M2NhLTYzMTItNjNhMC1iZmZmLTQ0NTZiM2JhOWRlM65jaGFubmVsX3ZhbHVlc4GpX19zdGFydF9fgahtZXNzYWdlc5HHrAWUvWxhbmdjaGFpbl9jb3JlLm1lc3NhZ2VzLmh1bWFurEh1bWFuTWVzc2FnZYanY29udGVudNkmSGkhIEknbSBBbGljZSBhbmQgSSBsaXZlIGluIEFtc3RlcmRhbS6xYWRkaXRpb25hbF9rd2FyZ3OAsXJlc3BvbnNlX21ldGFkYXRhgKR0eXBlpWh1bWFupG5hbWXAomlkwLNtb2RlbF92YWxpZGF0ZV9qc29usGNoYW5uZWxfdmVyc2lvbnOBqV9fc3RhcnRfXwGtdmVyc2lvbnNfc2VlboGpX19pbnB1dF9fgLB1cGRhdGVkX2NoYW5uZWxzkalfX3N0YXJ0X18=', 'type': 'msgpack', 'metadata': ['msgpack', 'g6Zzb3VyY2WlaW5wdXSkc3RlcP+ncGFyZW50c4A='], 'parent_checkpoint_id': '', '_rid': 'wT4SAIogbFwBAAAAAAAAAA==', '_self': 'dbs/wT4SAA==/colls/wT4SAIogbFw=/docs/wT4SAIogbFwBAAAAAAAAAA==/', '_etag': '"3f0ede32-0000-4700-0000-69f974ea0000"', '_attachments': 'attachments/', '_ts': 1777956074}